# Module 4: ARIMA End to End on One Agency

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

The four steps, in order, on one agency, with nothing skipped.

**Identify** the orders from the data rather than guessing them. **Estimate.**
**Diagnose**, and go back to step one if it fails. **Forecast**, and check the
forecast against the bar from Intermediate
[Module 13](../../Intermediate/Notebooks/Module_13_Baseline_Forecasts.ipynb).

The agency is Millgate, chosen because it has the weakest seasonal pattern in
the dataset, so a non seasonal model has a fair chance. [Module 5](Module_05_SARIMA.ipynb)
adds the seasonal orders.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]          # never fit on unfinished months


CALENDAR = pd.period_range("2019-01", "2026-04", freq="M").to_timestamp()


def counts(agency_id):
    """Monthly counts on a complete calendar, so a gap stays visible as missing."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series(d["n_uof"].values, dtype=float,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


print(f"{final['agency_id'].nunique()} agencies, {final['year_month'].nunique()} months")

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox


def ljung(resid, lag=12):
    return float(acorr_ljungbox(resid, lags=[lag], return_df=True)["lb_pvalue"].iloc[0])

s = np.log(counts("A004"))                 # Millgate
train, test = s.loc[:"2024-12"], s.loc["2025-01":"2025-12"]
print(f"train {len(train)} months, test {len(test)}, "
      f"mean {np.exp(train).mean():.1f} incidents a month")

## 2. Identify: how many times to difference

Run both tests from [Module 1](Module_01_Stationarity_Tested.ipynb) on the
level, then on the difference.

In [ ]:
def verdict(x, label):
    a = adfuller(x.dropna(), autolag="AIC")[1]
    k = kpss(x.dropna(), regression="c", nlags="auto")[1]
    v = ("stationary" if a < 0.05 <= k else
         "not stationary" if k < 0.05 <= a else
         "conflict" if a < 0.05 and k < 0.05 else "inconclusive")
    print(f"  {label:22s} ADF p={a:.3f}  KPSS p={k:.3f}  ->  {v}")


verdict(train, "level")
verdict(train.diff(), "first difference")

The level lands in the **conflict** cell: ADF rejects a unit root while KPSS
rejects stationarity. That combination usually means a trend with a stable part
around it, and the practical response is to difference once and check again,
which resolves it cleanly.

## 3. Identify: which orders

The two pictures name the model.

| Pattern | Reading |
|---|---|
| ACF cuts off sharply after lag q, PACF decays | a moving average term of order q |
| PACF cuts off sharply after lag p, ACF decays | an autoregressive term of order p |
| Both decay | possibly both, start small |

In [ ]:
d1 = train.diff().dropna()
band = 1.96 / np.sqrt(len(d1))
a_v, p_v = acf(d1, nlags=12, fft=False), pacf(d1, nlags=12)

print(f"noise band: plus or minus {band:.2f}\n")
print("  lag   ACF    PACF")
for k in range(1, 7):
    mark = "  <-" if abs(a_v[k]) > band or abs(p_v[k]) > band else ""
    print(f"  {k:3d}  {a_v[k]:+.2f}   {p_v[k]:+.2f}{mark}")

One large negative spike in the **ACF** at lag 1, and a **PACF** that decays
rather than cutting off. That is the signature of a moving average term of
order one, so the candidate is **ARIMA(0,1,1)**.

This is what identification means. The picture proposes a model; it does not
prove one, which is what the next two steps are for.

## 4. Estimate, and compare against the alternatives

In [ ]:
orders = [(0, 0, 0), (1, 0, 0), (2, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1), (0, 1, 1)]
rows = []
for o in orders:
    r = SARIMAX(train, order=o, seasonal_order=(0, 0, 0, 0),
                trend="c" if o[1] == 0 else "n",
                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    rows.append({"order": str(o), "AIC": round(r.aic, 1), "BIC": round(r.bic, 1),
                 "Ljung Box p": round(ljung(r.resid[3:]), 3),
                 "forecast error": round(float(np.mean(np.abs(
                     np.exp(test.values) - np.exp(r.forecast(12).values)))), 2)})

pd.DataFrame(rows).sort_values("AIC").set_index("order")

**The lowest AIC is `(0,1,1)`, exactly what the ACF and PACF proposed.** That
agreement is the point of doing identification at all: without it you are
searching blindly and the criterion has nothing to check.

Now read the last column, because it says something the first three do not.

In [ ]:
tbl = pd.DataFrame(rows).set_index("order")
baseline = float(np.mean(np.abs(np.exp(test.values) - np.exp(train.iloc[-12:].values))))
print(f"lowest AIC            : {tbl['AIC'].idxmin():10s} "
      f"forecast error {tbl.loc[tbl['AIC'].idxmin(), 'forecast error']:.2f}")
print(f"lowest forecast error : {tbl['forecast error'].idxmin():10s} "
      f"forecast error {tbl['forecast error'].min():.2f}")
print(f"the seasonal naive baseline            : {baseline:.2f}")
print(f"the series averages {np.exp(train).mean():.1f} incidents a month")

**The best fitting model is not the best forecasting model.** `(0,1,1)` wins on
AIC and a different order forecasts better, and both are within a fraction of
an incident of the constant only model on a series that averages under eight a
month.

The honest reading: every model here beats the seasonal naive baseline, and
none of them beats the others by anything a person could act on. That is the
right conclusion for a small, weakly structured series, and it is Intermediate
Module 13's lesson arriving with more machinery behind it.

**AIC ranks how well a model fits the data it was fitted to.** It is not a
forecast score, and the two answer different questions.

## 5. Diagnose

A model that fails here is not a model whose output you may quote, regardless
of its AIC.

In [ ]:
fit = SARIMAX(train, order=(0, 1, 1), seasonal_order=(0, 0, 0, 0)).fit(disp=False)
res = fit.resid[3:]
from scipy import stats

print(f"  Ljung Box at lag 12          p = {ljung(res, 12):.3f}")
print(f"  Ljung Box at lag 24          p = {ljung(res, 24):.3f}")
print(f"  Jarque Bera normality        p = {stats.jarque_bera(res)[1]:.3f}")
print(f"  squared residuals, lag 12    p = {ljung(res ** 2, 12):.3f}")
print(f"  largest standardised residual  {np.abs(res / res.std()).max():.2f}")

## 6. What the same model does to a seasonal series

Millgate was chosen because its season is weak. Try the same non seasonal
approach on Ashfell, where it is strong, and watch the diagnostics catch it.

In [ ]:
ash = np.log(counts("A012")).loc[:"2024-12"]
for label, o, so in [("ARIMA(1,1,1)", (1, 1, 1), (0, 0, 0, 0)),
                     ("ARIMA(2,1,2)", (2, 1, 2), (0, 0, 0, 0)),
                     ("SARIMA(0,1,1)(0,1,1)12", (0, 1, 1), (0, 1, 1, 12))]:
    r = SARIMAX(ash, order=o, seasonal_order=so, enforce_stationarity=False,
                enforce_invertibility=False).fit(disp=False)
    rr = r.resid[13:]
    print(f"  {label:24s} AIC {r.aic:7.1f}   "
          f"Ljung Box 12 p = {ljung(rr, 12):.4f}   24 p = {ljung(rr, 24):.4f}")

Both non seasonal models fail decisively, and **adding non seasonal complexity
makes it worse rather than better**: the more elaborate ARIMA(2,1,2) has a
smaller p value than ARIMA(1,1,1). Piling on autoregressive terms cannot
imitate a pattern that repeats every twelve months.

The seasonal model passes comfortably. That is [Module 5](Module_05_SARIMA.ipynb).

## 7. The workflow, as a checklist

| Step | What you are doing | What tells you to go back |
|---|---|---|
| Identify differencing | ADF and KPSS on the level, then the difference | still not stationary |
| Identify orders | ACF and PACF of the differenced series | nothing cuts off cleanly |
| Estimate | fit the candidate and its neighbours | the model does not converge |
| Diagnose | Ljung Box, normality, squared residuals, extreme points | any failure sends you to step one |
| Forecast | score against the baseline on held out data | no better than the baseline |

## Exercise

Run the workflow on Havenbrook. Does the ACF and PACF reading agree with the
AIC winner, as it did for Millgate?

In [ ]:
# Fill in the blank, then run.
AGENCY = None              # try "A003"

if AGENCY:
    x = np.log(counts(AGENCY))
    tr = x.loc[:"2024-12"]
    verdict(tr, "level"); verdict(tr.diff(), "first difference")
    d = tr.diff().dropna()
    b = 1.96 / np.sqrt(len(d))
    av, pv = acf(d, nlags=6, fft=False), pacf(d, nlags=6)
    print(f"\n  band +/-{b:.2f}")
    print("  ACF  1..6:", " ".join(f"{av[k]:+.2f}" for k in range(1, 7)))
    print("  PACF 1..6:", " ".join(f"{pv[k]:+.2f}" for k in range(1, 7)))
    out = []
    for o in [(0, 1, 1), (1, 1, 0), (1, 1, 1), (1, 0, 0), (0, 0, 0)]:
        r = SARIMAX(tr, order=o, trend="c" if o[1] == 0 else "n",
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        out.append({"order": str(o), "AIC": round(r.aic, 1),
                    "Ljung Box p": round(ljung(r.resid[3:]), 3)})
    print()
    print(pd.DataFrame(out).sort_values("AIC").to_string(index=False))
else:
    print("Set AGENCY above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A003"
```

**No, and the disagreement is the lesson.**

Havenbrook's **level** is already stationary: ADF gives p = 0.000 and KPSS
gives p = 0.100, the clean agreement that Millgate's level did not have. So
there is nothing to difference.

Follow Millgate's recipe anyway and you get `(0,1,1)` sitting **fifth of five**
on AIC, at 62.4 against 54.8 for the winner. The winner is `(1,0,0)`: an
autoregressive term on the **level**, no differencing at all.

Look at the differenced ACF to see why the recipe misfired. Millgate's lag one
was −0.55, a clear single spike. Havenbrook's is −0.26, barely outside the
band, with the PACF trailing off behind it. That is what **over differencing**
looks like from the inside: differencing a series that did not need it
manufactures a negative autocorrelation at lag 1, and a moving average term
then spends a parameter cancelling something the analyst created.

[Module 1](Module_01_Stationarity_Tested.ipynb) said differencing too much
raises the variance of what is left. This is the same warning arriving through
the model selection table instead. **Run the stationarity tests on the level
first, and if they agree it is stationary, do not difference.**

</details>

---

**Next:** [Module 5, SARIMA and Seasonal Orders](Module_05_SARIMA.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*